# Head Ablation Analysis: HML Benchmarks

**Attention head ablation results across:**
- 15 models (requested DeepSeek/Qwen/Llama/GPT-OSS/Gemma/Phi suite)
- 3 datasets: HumanEval, MBPP, LiveCodeBench
- 2 modes: Regular (CoT), Chain_Code

**Key Questions:**
1. Which attention heads are most critical for coding tasks?
2. How does head importance vary across benchmark datasets?
3. Are there universal critical heads across datasets?
4. How does chain_code mode differ from regular mode?

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

BASE_DIR = Path("../../results/hml")
REQUESTED_MODELS = [
    'Qwen--Qwen3-0.6B',
    'Qwen--Qwen3-4B',
    'Qwen--Qwen3-8B',
    'Qwen--Qwen3-14B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-7B',
    'deepseek-ai--DeepSeek-R1-Distill-Llama-8B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-14B',
    'deepseek-ai--DeepSeek-R1-0528-Qwen3-8B',
    'meta-llama--Llama-3.2-3B-Instruct',
    'openai--gpt-oss-20b',
    'google--gemma-4-E2B-it',
    'google--gemma-4-E4B-it',
    'google--gemma-4-26B-A4B-it',
    'microsoft--Phi-4-mini-reasoning',
]
MODEL_DIR_BY_REQUEST = {
    'Qwen--Qwen3-0.6B': 'Qwen3-0.6B',
    'Qwen--Qwen3-4B': 'Qwen3-4B',
    'Qwen--Qwen3-8B': 'Qwen3-8B',
    'Qwen--Qwen3-14B': 'Qwen3-14B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B': 'DeepSeek-R1-Distill-Qwen-1.5B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-7B': 'DeepSeek-R1-Distill-Qwen-7B',
    'deepseek-ai--DeepSeek-R1-Distill-Llama-8B': 'DeepSeek-R1-Distill-Llama-8B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-14B': 'DeepSeek-R1-Distill-Qwen-14B',
    'deepseek-ai--DeepSeek-R1-0528-Qwen3-8B': 'DeepSeek-R1-0528-Qwen3-8B',
    'meta-llama--Llama-3.2-3B-Instruct': 'Llama-3.2-3B-Instruct',
    'openai--gpt-oss-20b': 'gpt-oss-20b',
    'google--gemma-4-E2B-it': 'gemma-4-E2B-it',
    'google--gemma-4-E4B-it': 'gemma-4-E4B-it',
    'google--gemma-4-26B-A4B-it': 'gemma-4-26B-A4B-it',
    'microsoft--Phi-4-mini-reasoning': 'Phi-4-mini-reasoning',
}
MODELS = [MODEL_DIR_BY_REQUEST[m] for m in REQUESTED_MODELS]
MODEL_DISPLAY = {MODEL_DIR_BY_REQUEST[m]: m for m in REQUESTED_MODELS}
MODEL_SHORT = {
    'Qwen3-0.6B': 'Q3-0.6B',
    'Qwen3-4B': 'Q3-4B',
    'Qwen3-8B': 'Q3-8B',
    'Qwen3-14B': 'Q3-14B',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'DS-DQ1.5B',
    'DeepSeek-R1-Distill-Qwen-7B': 'DS-DQ7B',
    'DeepSeek-R1-Distill-Llama-8B': 'DS-DL8B',
    'DeepSeek-R1-Distill-Qwen-14B': 'DS-DQ14B',
    'DeepSeek-R1-0528-Qwen3-8B': 'DS-0528-Q8B',
    'Llama-3.2-3B-Instruct': 'Llama3.2-3B',
    'gpt-oss-20b': 'gpt-oss-20b',
    'gemma-4-E2B-it': 'Gemma4-E2B',
    'gemma-4-E4B-it': 'Gemma4-E4B',
    'gemma-4-26B-A4B-it': 'Gemma4-26B',
    'Phi-4-mini-reasoning': 'Phi4-mini',
}
DATASETS = ["humaneval", "mbpp", "livecodebench"]
MODES = {"CoT": "head_ablation_results", "Chain_Code": "head_ablation_results_chain_code"}

palette = sns.color_palette('tab20', n_colors=max(12, len(MODELS)))
MODEL_COLORS = {model: palette[i] for i, model in enumerate(MODELS)}
DS_COLORS = {"humaneval": "tab:green", "mbpp": "tab:purple", "livecodebench": "tab:red"}


def model_label(model, short=False):
    if short:
        return MODEL_SHORT.get(model, model)
    return MODEL_DISPLAY.get(model, model)


def grid_shape(n_items, max_cols=4):
    cols = min(max_cols, n_items)
    rows = int(np.ceil(n_items / cols))
    return rows, cols


def symmetric_percentile_limits(arrays, low=1, high=99):
    values = []
    for arr in arrays:
        if arr is None:
            continue
        arr = np.asarray(arr)
        finite = arr[np.isfinite(arr)]
        if finite.size == 0:
            continue
        values.append(np.percentile(finite, low))
        values.append(np.percentile(finite, high))

    if not values:
        return -1.0, 1.0

    max_abs_val = max(abs(min(values)), abs(max(values)))
    if not np.isfinite(max_abs_val) or max_abs_val == 0:
        max_abs_val = 1.0
    return -max_abs_val, max_abs_val

## 1. Data Loading

In [ ]:
def load_ablation_data(dataset, model, mode_dir):
    """Load ablation.npz and knockout.npz for a given dataset/model/mode."""
    ablation_path = BASE_DIR / dataset / mode_dir / model / dataset / "ablation.npz"
    knockout_path = BASE_DIR / dataset / mode_dir / model / dataset / "knockout.npz"
    if not ablation_path.exists():
        return None
    ablation = np.load(ablation_path)
    knockout = np.load(knockout_path) if knockout_path.exists() else None
    result = {
        "ablation_mean": ablation["ablation_mean"],
        "num_layers": ablation["ablation_mean"].shape[0],
        "num_heads": ablation["ablation_mean"].shape[1],
    }
    if knockout is not None:
        key = "knockout_losses" if "knockout_losses" in knockout else "avg_losses"
        if key in knockout:
            result["knockout_losses"] = knockout[key]
    return result

# Load all data
all_data = {}  # {mode: {model: {dataset: data}}}
for mode_name, mode_dir in MODES.items():
    all_data[mode_name] = {}
    for model in MODELS:
        all_data[mode_name][model] = {}
        print(f"=== {mode_name} | {model} ===")
        for ds in DATASETS:
            data = load_ablation_data(ds, model, mode_dir)
            if data:
                all_data[mode_name][model][ds] = data
                print(f"  {ds}: {data['num_layers']}L x {data['num_heads']}H")
            else:
                print(f"  {ds}: NOT FOUND")
        print()

## 2. Heatmaps: Ablation Matrices

In [ ]:
# Heatmaps for each model: datasets as columns, modes as rows
for model in MODELS:
    ablation_arrays = []
    for mode_name in MODES.keys():
        for ds in DATASETS:
            data = all_data[mode_name][model].get(ds)
            if data is not None:
                ablation_arrays.append(data['ablation_mean'])

    symmetric_vmin, symmetric_vmax = symmetric_percentile_limits(ablation_arrays)

    fig, axes = plt.subplots(len(MODES), len(DATASETS), figsize=(7 * len(DATASETS), 6 * len(MODES)), squeeze=False)
    mappable = None

    for row, (mode_name, mode_dir) in enumerate(MODES.items()):
        for col, ds in enumerate(DATASETS):
            ax = axes[row, col]
            data = all_data[mode_name][model].get(ds)

            if data is None:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center', fontsize=14)
            else:
                mappable = ax.imshow(
                    data['ablation_mean'],
                    aspect='auto',
                    cmap='coolwarm',
                    interpolation='nearest',
                    vmin=symmetric_vmin,
                    vmax=symmetric_vmax,
                )

            ax.set_xlabel('Head')
            ax.set_ylabel('Layer')
            ax.set_title(f'{ds} ({mode_name})', fontsize=12, fontweight='bold')

    fig.subplots_adjust(right=0.9)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    if mappable is not None:
        fig.colorbar(mappable, cax=cbar_ax)

    fig.suptitle(f'{model} - Ablation Importance Heatmaps', fontsize=15, y=1.02)
    plt.show()

## 3. Top Important Heads

In [ ]:
TOP_K = 20

top_heads_data = []
for mode_name in MODES:
    for model in MODELS:
        for ds in DATASETS:
            data = all_data[mode_name][model].get(ds)
            if data is None:
                continue
            ablation_matrix = data['ablation_mean']
            flat = ablation_matrix.flatten()
            top_k_indices = np.argsort(flat)[-TOP_K:][::-1]
            num_heads = data['num_heads']
            for rank, idx in enumerate(top_k_indices):
                l, h = idx // num_heads, idx % num_heads
                top_heads_data.append({
                    'Mode': mode_name, 'Model': model, 'Dataset': ds,
                    'Rank': rank+1, 'Layer': l, 'Head': h,
                    'Importance': flat[idx]
                })

top_heads_df = pd.DataFrame(top_heads_data)

# Show top 10 for each model (regular mode, averaged across datasets)
for model in MODELS:
    print(f"\n=== {model} (Regular mode, avg across datasets) ===")
    model_importances = {}
    for ds in DATASETS:
        data = all_data['CoT'][model].get(ds)
        if data is None:
            continue
        mat = data['ablation_mean']
        for l in range(mat.shape[0]):
            for h in range(mat.shape[1]):
                key = (l, h)
                model_importances.setdefault(key, []).append(mat[l, h])
    avg_imp = {k: np.mean(v) for k, v in model_importances.items()}
    ranked = sorted(avg_imp.items(), key=lambda x: x[1], reverse=True)[:10]
    print(f"{'Rank':>4} | {'Layer':>5} | {'Head':>4} | {'Mean dLoss':>12}")
    print('-'*40)
    for i, ((l, h), imp) in enumerate(ranked):
        print(f"{i+1:4d} | {l:5d} | {h:4d} | {imp:12.4f}")

## 4. Layer-wise Importance Distribution

## 5. Top Head Consistency Across Datasets

In [ ]:
K_STABLE = 10

def get_top_k_heads(mode_name, model, ds, k=10):
    data = all_data[mode_name][model].get(ds)
    if data is None:
        return set()
    flat = data['ablation_mean'].flatten()
    top_k_idx = np.argsort(flat)[-k:][::-1]
    num_heads = data['num_heads']
    return {(idx // num_heads, idx % num_heads) for idx in top_k_idx}

for mode_name in MODES:
    for model in MODELS:
        print(f"\n=== {model} ({mode_name}) | Top-{K_STABLE} Jaccard similarity ===")
        ds_heads = {ds: get_top_k_heads(mode_name, model, ds, K_STABLE) for ds in DATASETS}
        n = len(DATASETS)
        jaccard = np.zeros((n, n))
        for i, di in enumerate(DATASETS):
            for j, dj in enumerate(DATASETS):
                inter = len(ds_heads[di] & ds_heads[dj])
                union = len(ds_heads[di] | ds_heads[dj])
                jaccard[i, j] = inter / union if union > 0 else 0

        symmetric_vmin, symmetric_vmax = symmetric_percentile_limits([jaccard])

        fig, ax = plt.subplots(figsize=(6, 5))
        im = ax.imshow(jaccard, cmap='coolwarm', vmin=symmetric_vmin, vmax=symmetric_vmax)
        ax.set_xticks(range(n)); ax.set_yticks(range(n))
        ax.set_xticklabels(DATASETS); ax.set_yticklabels(DATASETS)
        for i in range(n):
            for j in range(n):
                ax.text(j, i, f'{jaccard[i,j]:.2f}', ha='center', va='center', fontsize=12, fontweight='bold')
        plt.colorbar(im, ax=ax)
        ax.set_title(f'{model} ({mode_name}) - Top-{K_STABLE} Jaccard', fontsize=13)
        plt.show()

## 6. CoT vs Chain_Code Comparison

In [ ]:
# Side-by-side heatmaps: CoT vs Chain_Code for each model+dataset
for model in MODELS:
    heatmap_arrays = []
    for ds in DATASETS:
        cot_data = all_data['CoT'][model].get(ds)
        cc_data = all_data['Chain_Code'][model].get(ds)
        if cot_data is not None:
            heatmap_arrays.append(cot_data['ablation_mean'])
        if cc_data is not None:
            heatmap_arrays.append(cc_data['ablation_mean'])
        if cot_data is not None and cc_data is not None:
            heatmap_arrays.append(cc_data['ablation_mean'] - cot_data['ablation_mean'])

    symmetric_vmin, symmetric_vmax = symmetric_percentile_limits(heatmap_arrays)

    fig, axes = plt.subplots(len(DATASETS), 3, figsize=(21 * 0.95, 6 * len(DATASETS)), squeeze=False)
    mappable = None

    for row, ds in enumerate(DATASETS):
        cot_data = all_data['CoT'][model].get(ds)
        cc_data = all_data['Chain_Code'][model].get(ds)

        for col, (mode_name, data) in enumerate([('CoT', cot_data), ('Chain_Code', cc_data)]):
            ax = axes[row, col]
            if data is None:
                ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            else:
                mappable = ax.imshow(
                    data['ablation_mean'],
                    aspect='auto',
                    cmap='coolwarm',
                    vmin=symmetric_vmin,
                    vmax=symmetric_vmax,
                )
            ax.set_title(f'{ds} - {mode_name}', fontsize=12)
            ax.set_xlabel('Head'); ax.set_ylabel('Layer')

        # Difference heatmap
        ax = axes[row, 2]
        if cot_data is not None and cc_data is not None:
            diff = cc_data['ablation_mean'] - cot_data['ablation_mean']
            mappable = ax.imshow(
                diff,
                aspect='auto',
                cmap='coolwarm',
                vmin=symmetric_vmin,
                vmax=symmetric_vmax,
            )
        else:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
        ax.set_title(f'{ds} - Diff (CC - CoT)', fontsize=12)
        ax.set_xlabel('Head'); ax.set_ylabel('Layer')

    fig.subplots_adjust(right=0.9)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    if mappable is not None:
        fig.colorbar(mappable, cax=cbar_ax)

    fig.suptitle(f'{model} - CoT vs Chain_Code', fontsize=15, y=1.02)
    plt.show()

## 7. Progressive Knockout Analysis

In [ ]:
for mode_name in MODES:
    rows, cols = grid_shape(len(MODELS), max_cols=4)
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows), squeeze=False)

    for m_idx, model in enumerate(MODELS):
        r, c = divmod(m_idx, cols)
        ax = axes[r, c]
        for ds in DATASETS:
            data = all_data[mode_name][model].get(ds)
            if data is None or 'knockout_losses' not in data:
                continue
            losses = data['knockout_losses']
            x = np.arange(len(losses))
            ax.plot(x, losses, label=ds, color=DS_COLORS[ds], linewidth=2)
        ax.set_xlabel('Heads Knocked Out', fontsize=11)
        ax.set_ylabel('Loss', fontsize=11)
        ax.set_title(model_label(model, short=True), fontsize=12)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    total_slots = rows * cols
    for empty_idx in range(len(MODELS), total_slots):
        r, c = divmod(empty_idx, cols)
        axes[r, c].axis('off')

    fig.suptitle(f'Progressive Knockout ({mode_name})', fontsize=15, y=1.01)
    fig.tight_layout()
    plt.show()

## 8. Cross-Model Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Configuration ---
# Any model with a mean head importance HIGHER than this in ANY dataset will be hidden in the zoomed row
ZOOM_MAX_THRESHOLD = 0.05 
# ---------------------

for mode_name in MODES:
    fig, axes = plt.subplots(2, len(DATASETS), figsize=(8 * len(DATASETS), 12), squeeze=False)
    
    # 1. Pre-calculate which models are allowed in the zoomed (bottom) row.
    # A model is allowed ONLY if it stays <= ZOOM_MAX_THRESHOLD across ALL datasets.
    allowed_in_zoom = set(MODELS)
    for model in MODELS:
        for ds in DATASETS:
            data = all_data[mode_name][model].get(ds)
            if data is not None:
                layer_avg = data['ablation_mean'].mean(axis=1)
                if mode_name == "Chain_Code":
                    ZOOM_MAX_THRESHOLD = 0.02
                if np.any(layer_avg > ZOOM_MAX_THRESHOLD):
                    allowed_in_zoom.discard(model)
                    break # Skip checking other datasets for this model, it's already disqualified
                    
    for d_idx, ds in enumerate(DATASETS):
        ax_orig = axes[0, d_idx]
        ax_zoom = axes[1, d_idx]
        
        skip_layers = 3  # Keep your original early-layer skip for the bottom row
        
        for model in MODELS:
            data = all_data[mode_name][model].get(ds)
            if data is None:
                continue
            
            layer_avg = data['ablation_mean'].mean(axis=1)
            x = np.linspace(0, 1, len(layer_avg))
            label = model_label(model, short=True)
            
            # --- TOP ROW: No filters, plot all models ---
            ax_orig.plot(x, layer_avg, label=label, color=MODEL_COLORS[model], linewidth=2)
            
            # --- BOTTOM ROW: Filtered models only ---
            if model in allowed_in_zoom and len(layer_avg) > skip_layers:
                ax_zoom.plot(x[skip_layers:], layer_avg[skip_layers:],
                             label=label, color=MODEL_COLORS[model], linewidth=2)

        # Format original axes
        ax_orig.set_xlabel('Normalized Layer Position', fontsize=12)
        ax_orig.set_ylabel('Mean Head Importance', fontsize=12)
        ax_orig.set_title(f'{ds} (All Models)', fontsize=13)
        ax_orig.grid(True, alpha=0.3)
        
        # Format zoomed axes
        ax_zoom.set_xlabel('Normalized Layer Position', fontsize=12)
        ax_zoom.set_ylabel('Mean Head Importance', fontsize=12)
        ax_zoom.set_title(f'{ds} (Zoomed & Filtered < {ZOOM_MAX_THRESHOLD})', fontsize=13)
        ax_zoom.grid(True, alpha=0.3)

    # Global legend and layout
    # Grab labels from the top row so the legend includes all models
    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.01, 0.5), title='Model')
        
    fig.suptitle(f'Cross-Model Layer Importance ({mode_name})', fontsize=15, y=0.99)
    fig.tight_layout(rect=[0, 0, 0.84, 0.98])
    plt.show()

## 9. Summary Statistics

In [ ]:
summary_data = []
for mode_name in MODES:
    for model in MODELS:
        for ds in DATASETS:
            data = all_data[mode_name][model].get(ds)
            if data is None:
                continue
            flat = data['ablation_mean'].flatten()
            row = {
                'Mode': mode_name, 'Model': model, 'Dataset': ds,
                'Max Importance': f'{flat.max():.4f}',
                'Mean Importance': f'{flat.mean():.4f}',
                'Std Importance': f'{flat.std():.4f}',
                'Top-1 Head': f'L{np.unravel_index(flat.argmax(), data["ablation_mean"].shape)[0]}H{np.unravel_index(flat.argmax(), data["ablation_mean"].shape)[1]}',
            }
            if 'knockout_losses' in data:
                ko = data['knockout_losses']
                row['Baseline Loss'] = f'{ko[0]:.4f}'
                row['Loss @10 KO'] = f'{ko[min(10, len(ko)-1)]:.4f}'
            summary_data.append(row)

df = pd.DataFrame(summary_data)
print(df.to_string(index=False))